In [ ]:
!git clone -b ml-lr https://github.com/JKaraman93/bet.git

In [ ]:
cd ..

In [ ]:
from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import src.utils.config as config


In [ ]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()
player_behavior = spark.read.parquet("./data/gold/player_behavior")
player_snapshot = spark.read.parquet("./data/gold/player_snapshot")
labels = spark.read.parquet("./data/gold/labels")


In [ ]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegression

In [ ]:

player_snapshot = player_snapshot.select('player_idx',
 'country',
 'age_bucket',
 #'device_type',
 #'acquisition_channel',
 #'registration_date',
 #'risk_segment',
 )

model_df = (
    player_behavior
        .join(
            player_snapshot,
            on="player_idx",
            how="left"
        )
        .join(
            labels,
            on=["player_idx", "reference_date"],
            how="inner"   # label must exist
        )
)



In [ ]:
sizeOfDataset = model_df.count()
sizeOfDataset

In [ ]:
sample_dataset = (player_snapshot.select('player_idx')
                .sample(0.1)
                .join(model_df, how='inner', on='player_idx')

)
sample_dataset.count()

In [ ]:
numeric_cols = [
    "balance_7d_ago", "balance_30d_ago",
    "net_amount_result_7d", "net_amount_result_30d",
    "num_sessions_7d", "num_sessions_30d",
    "avg_sessions_duration_30d",
    "avg_bet_amount_30d",
    "net_game_result_7d", "net_game_result_30d",
    "failed_withdrawals_30d",
    "deposit_count_30d", "withdrawal_count_30d",
    "withdrawal_ratio"
] 

categorical_cols = [
    "country",
    "age_bucket",
]


In [ ]:
categorical_cols_idx = [c + '_idx' for c in categorical_cols]
categorical_cols_ohe = [c + '_ohe' for c in categorical_cols]
numeric_cols_sc = [c + '_sc' for c in numeric_cols]


In [ ]:
new_df = sample_dataset.withColumn('next_7d_churn_idx', F.col('next_7d_churn').cast('int'))

In [ ]:
indexers =  StringIndexer(
        inputCols=categorical_cols,
        outputCols=categorical_cols_idx,
        handleInvalid="error"  #keep
    )

new_df = indexers.fit(new_df).transform(new_df)

ohe = OneHotEncoder(
    inputCols=categorical_cols_idx,
    outputCols=categorical_cols_ohe,
        dropLast=False
)

new_df2 = ohe.fit(new_df).transform(new_df)
new_df2.show()

In [ ]:
numeric_assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="numeric_features"
)

new_df3 = numeric_assembler.transform(new_df2)
new_df3.show()

In [ ]:
std_sc = StandardScaler(
    inputCol='numeric_features',
    outputCol='numeric_features_scaled',
    withMean=True,
    withStd=True
)


new_df4 = std_sc.fit(new_df3).transform(new_df3)
new_df4.show()

In [ ]:
final_assembler = VectorAssembler(
    inputCols=["numeric_features_scaled"] + categorical_cols_ohe,
    outputCol="features"
)
new_df5 = final_assembler.transform(new_df4)
new_df5.show()

In [ ]:
lr = LogisticRegression(
    featuresCol='features',
    labelCol='next_7d_churn_idx',
       maxIter=50,
    regParam=0.01, 
        elasticNetParam=0.0   # pure L2 to start
)


In [ ]:
model_df = new_df5

In [ ]:
dates = (
    model_df
    .select("reference_date")
    .distinct()
    .orderBy("reference_date")
    .collect()
)

n = len(dates)
train_cut = dates[int(n * 0.70)][0]
val_cut   = dates[int(n * 0.85)][0]

train_df = model_df.filter(F.col("reference_date") < train_cut)
val_df   = model_df.filter((F.col("reference_date") >= train_cut) & (F.col("reference_date") < val_cut))
test_df  = model_df.filter(F.col("reference_date") >= val_cut)


In [ ]:
model = lr.fit(train_df)


In [32]:
preds = model.transform(val_df)


In [38]:
from pyspark.ml.functions import vector_to_array

preds = preds.withColumn(
    "p_churn",
    vector_to_array("probability")[1]
)
preds = preds.withColumn(
    "pred_label",
    (F.col("p_churn") >= 0.1).cast("int")
)
preds.show()


+----------+--------------+------------------+------------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+-------+----------+-------------+-----------------+-----------+--------------+-------------+--------------+--------------------+-----------------------+--------------------+--------------------+--------------------+----------+--------------------+----------+
|player_idx|reference_date|    balance_7d_ago|   balance_30d_ago|net_amount_result_7d|net_amount_result_30d|num_sessions_7d|net_game_result_7d|num_sessions_30d|avg_sessions_duration_30d|avg_bet_amount_30d|net_game_result_30d|failed_withdrawals_30d|deposit_count_30d|withdrawal_count_30d|withdrawal_ratio|country|age_bucket|next_7d_churn|next_7d_churn_idx|country_idx|age_bucket_idx|  country_ohe|age_bucket_ohe|    numeric_features|numer

In [ ]:

evaluator = BinaryClassificationEvaluator(
    labelCol="next_7d_churn_idx",
    metricName="areaUnderROC"
)

auc = evaluator.evaluate(preds)
print(f"Validation AUC: {auc:.4f}")

Validation AUC: 0.8594


In [39]:
confusion = (
    preds
    .groupBy("next_7d_churn_idx", "pred_label")
    .count()
    .orderBy("next_7d_churn_idx", "pred_label")
)
confusion.show()
cm = {
    (row["next_7d_churn_idx"], row["pred_label"]): row["count"]
    for row in confusion.collect()
}

TN = cm.get((0, 0), 0)
FP = cm.get((0, 1), 0)
FN = cm.get((1, 0), 0)
TP = cm.get((1, 1), 0)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")

+-----------------+----------+-----+
|next_7d_churn_idx|pred_label|count|
+-----------------+----------+-----+
|                0|         0|90380|
|                0|         1|21782|
|                1|         0| 3521|
|                1|         1|11446|
+-----------------+----------+-----+



Precision: 0.344
Recall:    0.765
F1-score:  0.475


26/01/26 16:48:40 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 510249 ms exceeds timeout 120000 ms
26/01/26 16:48:40 WARN SparkContext: Killing executors is not supported by current scheduler.
26/01/26 16:48:43 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1324)
	at o

In [ ]:
preds.show()